<a href="https://colab.research.google.com/github/EstherMan05/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import os, re
import pandas as pd

CANDIDATES = [
    'product_catalog.csv',
    '/content/product_catalog.csv',
    '../06-midterm-walmart/data/product_catalog.csv',
]
PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert PATH is not None, "product_catalog.csv not found - upload it in the Files panel, then re-run."

cat = pd.read_csv(PATH)
print('loaded real catalog:', cat.shape, 'from', PATH)
cat.head()

loaded real catalog: (45, 7) from product_catalog.csv


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
is_pop = cat['product_name'].str.lower().str.contains('pop', na=False)
pop = cat[is_pop]
print('pop-tart-like rows:', len(pop))
pop

pop-tart-like rows: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
straw = pop[pop['product_name'].str.lower().str.contains('straw', na=False)]
other_pop = pop[~pop.index.isin(straw.index)]

print('Strawberry variants:', len(straw))
print(straw[['sku', 'product_name']].to_string(index=False))
print()
print('Pop-Tart rows that are NOT strawberry:', len(other_pop))
other_pop

Strawberry variants: 4
          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry
PT_STRAWBERRY pop-tarts strawberry

Pop-Tart rows that are NOT strawberry: 0


,sku,product_name,category,subcategory,unit_cost,supplier,package_size


**Pop-Tart rows that are a different product:** None in this catalog. All four Pop-Tart rows (`PT-12`, `SKU#459812`, `POPTART-STRAW`, `PT_STRAWBERRY`) are strawberry under different SKUs and spellings. Another flavor such as blueberry would have to stay separate, because a different flavor is a different product.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
rows_before = len(cat)

CANON = {sku: 'POPTART-STRAW' for sku in straw['sku']}
print('mapping:', CANON)

# .map() gives NaN for unmapped SKUs, so fillna with the original SKU keeps them as themselves.
cat['canonical_sku'] = cat['sku'].map(CANON).fillna(cat['sku'])

print('distinct SKUs:', cat['sku'].nunique(), '-> distinct canonical SKUs:', cat['canonical_sku'].nunique())
cat[cat['canonical_sku'] == 'POPTART-STRAW'][['sku', 'product_name', 'canonical_sku']]

mapping: {'PT-12': 'POPTART-STRAW', 'SKU#459812': 'POPTART-STRAW', 'POPTART-STRAW': 'POPTART-STRAW', 'PT_STRAWBERRY': 'POPTART-STRAW'}
distinct SKUs: 45 -> distinct canonical SKUs: 42


,sku,product_name,canonical_sku
0,PT-12,pop-tarts strawberry,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,POPTART-STRAW
3,PT_STRAWBERRY,pop-tarts strawberry,POPTART-STRAW


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
raw_cat = pd.read_csv(PATH)['category']

# BEFORE: raw labels with counts
print('BEFORE:', raw_cat.nunique(), 'distinct labels')
print(raw_cat.value_counts().to_string())

# Strip whitespace + title-case
step1 = raw_cat.str.strip().str.title()
print('\nAFTER strip + title-case:', step1.nunique(), 'distinct labels')
print(step1.value_counts().to_string())

# Merge labels that are still two names for one category
CAT_MAP = {
    'Breakfast Foods': 'Breakfast',
    'Emergency Supplies': 'Emergency',
    'Alcoholic Beverages': 'Alcohol',
    'Groceries': 'Grocery',
    'Baby Products': 'Baby',
}
cat['category'] = step1.replace(CAT_MAP)
print('\nAFTER mapping:', cat['category'].nunique(), 'distinct labels')
print(cat['category'].value_counts().to_string())

BEFORE: 27 distinct labels
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1

AFTER strip + title-case: 15 distinct labels
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health   

Title-casing only fixes capitalization (27 labels down to 15). These pairs are still two labels for one category: `Breakfast` / `Breakfast Foods`, `Emergency` / `Emergency Supplies`, `Grocery` / `Groceries`, `Baby` / `Baby Products`, and `Alcohol` / `Alcoholic Beverages`. I merged them by hand, which gives 10 categories. I left `Snacks` separate from `Grocery`, and `Health` separate from `Household`, because whether those are the same category is a business decision, not a spelling fix.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
def name_key(s):
    """lowercase, strip punctuation, crude singularize, sort the words"""
    words = re.sub(r'[^a-z0-9\s]', ' ', str(s).lower()).split()
    words = [w[:-1] if w.endswith('s') and len(w) > 3 else w for w in words]
    return ' '.join(sorted(words))

cat['name_key'] = cat['product_name'].map(name_key)
skus_per_key = cat.groupby('name_key')['sku'].nunique()
fragmented = skus_per_key[skus_per_key > 1].sort_values(ascending=False)

print('candidate groups with more than one SKU:', len(fragmented))
for key in fragmented.index:
    g = cat[cat['name_key'] == key]
    print(f"\n[{key}]  ({len(g)} SKUs)")
    print(g[['sku', 'product_name', 'unit_cost', 'package_size']].to_string(index=False))

candidate groups with more than one SKU: 14

[pop strawberry tart]  (4 SKUs)
          sku         product_name  unit_cost package_size
        PT-12 pop-tarts strawberry       1.93       4-pack
   SKU#459812  Strawberry Pop-Tart       2.26          4pk
POPTART-STRAW Pop-Tarts Strawberry       2.11       6 pack
PT_STRAWBERRY pop-tarts strawberry        NaN       4 pack

[board game monopoly]  (3 SKUs)
        sku          product_name  unit_cost package_size
    BG-MONO   Monopoly Board Game       9.96          1pk
MONOPOLY-01   Monopoly Board Game      12.51      24-pack
  GAME-MONO Board Game - Monopoly        NaN          1pk

[4 aa batterie pack]  (2 SKUs)
        sku        product_name  unit_cost package_size
    BAT-AA4 AA Batteries 4-Pack       4.16          4pk
BATT_AA_4PK AA Batteries 4-Pack       3.71      24 pack

[aid first kit]  (2 SKUs)
         sku  product_name  unit_cost package_size
     FAID-01 First-Aid Kit       6.90       4 pack
FIRSTAID-KIT First Aid Kit       7

In [7]:
for kw in ['water', 'batter']:
    g = cat[cat['product_name'].str.lower().str.contains(kw, na=False)]
    print(f"\n--- '{kw}' ({len(g)} rows, {g['name_key'].nunique()} distinct name keys) ---")
    print(g[['sku', 'product_name', 'name_key', 'package_size']].to_string(index=False))


--- 'water' (3 rows, 3 distinct name keys) ---
      sku          product_name              name_key package_size
   WTR-24    BOTTLED WATER 24PK    24pk bottled water       1 pack
WATER24PK Bottled Water 24-Pack 24 bottled pack water      24 pack
 WTR_24PK       Water - 24 Pack         24 pack water       1 pack

--- 'batter' (3 rows, 2 distinct name keys) ---
        sku        product_name           name_key package_size
   BATT-100    BATTERIES AA 4PK    4pk aa batterie       6 pack
    BAT-AA4 AA Batteries 4-Pack 4 aa batterie pack          4pk
BATT_AA_4PK AA Batteries 4-Pack 4 aa batterie pack      24 pack


The name key is only a candidate list. It found 14 groups, but it misses families whose wording differs too much (bottled water, and one of the three AA battery SKUs), so I also scan by hand.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
assert cat['canonical_sku'].notna().all(), 'canonical_sku has nulls'
assert len(cat) == rows_before, 'row count changed'
assert cat['category'].notna().all(), 'category has nulls'
print(f'mapping is safe: {len(cat)} rows, no nulls in canonical_sku')

mapping is safe: 45 rows, no nulls in canonical_sku


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A) Two families beyond Pop-Tarts look fragmented. Monopoly Board Game appears under three SKUs (BG-MONO, MONOPOLY-01, GAME-MONO) with names like “Monopoly Board Game” and “Board Game - Monopoly”; the Q5 name key caught these because the words are the same in a different order. Bottled water appears under three SKUs (WTR-24, WATER24PK, WTR_24PK) with names like “BOTTLED WATER 24PK” and “Water - 24 Pack”; the Q5 key missed this family because the wording differs too much, so I found it by scanning for “water”. I would map each family’s variants to one canonical SKU and leave every unmapped SKU as itself. The Q5 result is only a candidate list: it found 14 groups but missed others (such as the AA batteries), so I would check each family by hand before merging.

B) Consolidation stops being data cleaning when the question changes from “is this the same string?” to “is this the same product for the business?” Merging “pop-tarts strawberry” and “Pop-Tarts Strawberry” is a spelling fix. The Pop-Tart family is where I am genuinely unsure: POPTART-STRAW is listed as a 6 pack while the other three are 4-packs, and unit_cost ranges from 1.93 to 2.26 across them. If they are different pack sizes, merging would mix units and costs that are not comparable. The supplier column is also inconsistent across these rows (it even lists a streaming-device company on a Pop-Tart row), so I would not rely on it to decide. I would ask the category manager or whoever owns the product hierarchy whether pack size makes a separate product, and I would flag it to my instructor or TA before the midterm.